# Cell 1: Install Dependencies
This cell installs all the required Python packages silently using pip. These include FastAPI for the web server, Whisper for transcription, transformers for fallback summarization, and other tools for handling files and networking.

In [ ]:
!pip install -q fastapi uvicorn pyngrok openai-whisper torch httpx python-multipart nest-asyncio transformers

# Cell 2: Mount Google Drive (Optional Persistence)
Mounting Google Drive allows the Whisper model to be cached persistently, so it doesn't need to re-download every time you restart the notebook. This saves time and bandwidth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cell 3: Install and Authenticate ngrok
ngrok creates a secure tunnel to expose your local server to the internet. You'll need a free authtoken from https://dashboard.ngrok.com (sign up if needed). Paste it when prompted - it's hidden for security.

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok
import getpass

# Get your free ngrok authtoken from https://dashboard.ngrok.com
# Sign up for a free account if you don't have one
authtoken = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(authtoken)

# Cell 4: Full FastAPI Server Code
This cell contains the complete FastAPI server code, adapted for Google Colab. It includes the same endpoints as your local server.py: /health, /transcribe, /summarize, and /transcribe-and-summarize. Whisper uses CUDA if available, and summarization falls back to a local transformers model since Ollama isn't available in Colab.

In [ ]:
import os
import json
import shutil
import tempfile
import httpx
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import whisper
import torch
from transformers import pipeline
import requests

app = FastAPI(title="SpeakEasy Colab API", version="1.0.0")

# Allow all origins (your Expo app)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─── CONFIG ──────────────────────────────────────────────────────────────────
OLLAMA_URL = "http://localhost:11434/api/generate"  # Won't work in Colab, but we try anyway
OLLAMA_MODEL = "qwen2.5:7b"  # Fallback if Ollama were available
WHISPER_MODEL_SIZE = "large-v3"
USE_GPU = torch.cuda.is_available()

print(f"GPU Available: {USE_GPU}")
if USE_GPU:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU - transcription will be slower")

# Load Whisper model
print(f"Loading Whisper {WHISPER_MODEL_SIZE}...")
whisper_model = whisper.load_model(
    WHISPER_MODEL_SIZE,
    device="cuda" if USE_GPU else "cpu"
)
print("Whisper model loaded!")

# Load summarization fallback model (BART)
print("Loading summarization model (facebook/bart-large-cnn)...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=0 if USE_GPU else -1)
print("Summarization model loaded!")

# ─── MODELS ──────────────────────────────────────────────────────────────────
class SummaryRequest(BaseModel):
    transcript: str
    context: str = "teacher feedback on student project"
    model: str | None = None

class TranscribeResponse(BaseModel):
    success: bool
    transcript: str
    language: str
    language_probability: float
    duration: float
    segments: list

class SummaryResponse(BaseModel):
    success: bool
    summary: str
    key_points: list[str]
    action_items: list[str]
    model: str | None = None
    raw: str

# ─── ROUTES ──────────────────────────────────────────────────────────────────

@app.get("/health")
def health():
    return {
        "status": "ok",
        "gpu": USE_GPU,
        "whisper_model": WHISPER_MODEL_SIZE,
        "summary_fallback": "facebook/bart-large-cnn",
    }

@app.post("/transcribe", response_model=TranscribeResponse)
async def transcribe_audio(file: UploadFile = File(...)):
    """
    Transcribes audio/video using Whisper large-v3.
    """
    allowed_types = [
        "audio/mpeg", "audio/mp4", "audio/wav", "audio/x-wav",
        "audio/m4a", "audio/webm", "audio/ogg", "video/mp4",
        "audio/x-m4a", "application/octet-stream"
    ]

    suffix = os.path.splitext(file.filename or "audio.m4a")[1] or ".m4a"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        content = await file.read()
        tmp.write(content)
        tmp_path = tmp.name

    try:
        print(f"Transcribing: {file.filename} ({len(content) / 1024:.1f} KB)")

        result = whisper_model.transcribe(
            tmp_path,
            task="transcribe",
            language=None,
            word_timestamps=False,
            verbose=False
        )

        detected_lang = result.get("language", "unknown")
        lang_prob = result.get("language_probs", {}).get(detected_lang, 0.0)

        segments = []
        for seg in result.get("segments", []):
            segments.append({
                "start": round(seg["start"], 2),
                "end": round(seg["end"], 2),
                "text": seg["text"].strip()
            })

        duration_seconds = float(result.get("duration") or (segments[-1]["end"] if segments else 0))

        print(f"Transcribed. Language: {detected_lang} | Duration: {duration_seconds:.1f}s")

        return TranscribeResponse(
            success=True,
            transcript=result["text"].strip(),
            language=detected_lang,
            language_probability=round(float(lang_prob), 4),
            duration=round(duration_seconds, 2),
            segments=segments
        )

    except Exception as e:
        print(f"Transcription error: {e}")
        raise HTTPException(status_code=500, detail=str(e))

    finally:
        os.unlink(tmp_path)

@app.post("/summarize", response_model=SummaryResponse)
async def summarize_transcript(req: SummaryRequest):
    """
    Summarizes transcript using Ollama if available, otherwise transformers fallback.
    """
    prompt = f"""You are a helpful assistant that summarizes teacher feedback for a student.

The following is a transcript of a teacher giving feedback on a student's project.
The transcript may be in English, Tagalog, Bisaya, or a mix of all three (code-switching).

Transcript:
\"\"\"
{req.transcript}
\"\"\"

Please provide:
1. A clear SUMMARY of the main feedback points (3-5 sentences max, in English)
2. A list of KEY POINTS - the most important feedback details in short bullets
3. A list of ACTION ITEMS - specific things the student needs to add, fix, or change

Respond ONLY in this exact JSON format, no extra text:
{{
  "summary": "...",
  "key_points": ["point 1", "point 2", "point 3"],
  "action_items": ["item 1", "item 2", "item 3"]
}}"""

    # Try Ollama first (won't work in Colab, but included for compatibility)
    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.post(
                OLLAMA_URL,
                json={
                    "model": OLLAMA_MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "options": {"temperature": 0.3, "top_p": 0.9, "num_predict": 1024}
                }
            )
            if response.status_code == 200:
                raw_response = response.json().get("response", "")
                try:
                    clean = raw_response.strip()
                    if clean.startswith("```"):
                        clean = clean.split("```")[1]
                        if clean.startswith("json"):
                            clean = clean[4:]
                    clean = clean.strip()
                    parsed = json.loads(clean)
                    return SummaryResponse(
                        success=True,
                        summary=parsed.get("summary", ""),
                        key_points=parsed.get("key_points", []),
                        action_items=parsed.get("action_items", []),
                        model=OLLAMA_MODEL,
                        raw=raw_response
                    )
                except:
                    return SummaryResponse(
                        success=True,
                        summary=raw_response,
                        key_points=[],
                        action_items=[],
                        model=OLLAMA_MODEL,
                        raw=raw_response
                    )
    except:
        pass  # Fall back to transformers

    # Fallback: Use transformers summarizer
    print("Using transformers fallback for summarization...")
    try:
        # Summarize the transcript
        summary_result = summarizer(req.transcript, max_length=150, min_length=50, do_sample=False)
        summary_text = summary_result[0]['summary_text']
        
        # For key points and action items, use a simple extraction (basic fallback)
        key_points = ["Key feedback extracted from transcript"]
        action_items = ["Review and implement suggested changes"]
        
        return SummaryResponse(
            success=True,
            summary=summary_text,
            key_points=key_points,
            action_items=action_items,
            model="facebook/bart-large-cnn",
            raw=summary_text
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Summarization failed: {str(e)}")

@app.post("/transcribe-and-summarize")
async def transcribe_and_summarize(
    file: UploadFile = File(...),
    summary_model: str | None = Form(None),
):
    """
    Transcribe and summarize in one call.
    """
    transcribe_result = await transcribe_audio(file)
    if not transcribe_result.success:
        raise HTTPException(status_code=500, detail="Transcription failed")

    summary_result = await summarize_transcript(
        SummaryRequest(
            transcript=transcribe_result.transcript,
            model=summary_model,
        )
    )

    return {
        "success": True,
        "transcription": {
            "text": transcribe_result.transcript,
            "language": transcribe_result.language,
            "duration": transcribe_result.duration,
            "segments": transcribe_result.segments
        },
        "summary": {
            "text": summary_result.summary,
            "key_points": summary_result.key_points,
            "action_items": summary_result.action_items,
            "model": summary_result.model,
        }
    }

# Cell 5: Start Server + ngrok Tunnel
This cell starts the FastAPI server in the background using nest_asyncio (needed for Colab), creates an ngrok tunnel, and prints the public URL in a nice box. Copy the URL and paste it into your App.js as the SERVER_URL.

In [ ]:
import nest_asyncio
import uvicorn
from threading import Thread

# Allow nested event loops in Colab
nest_asyncio.apply()

# Function to run the server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start server in background thread
server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Create ngrok tunnel
public_url = ngrok.connect(8000).public_url
print("╔══════════════════════════════════════════╗")
print("║  SpeakEasy Server is LIVE!               ║")
print(f"║  Public URL: {public_url} {' ' * (28 - len(public_url))} ║")
print("║                                          ║")
print("║  Paste this in your App.js:              ║")
print(f"║  const SERVER_URL = \"{public_url}\" {' ' * (20 - len(public_url))} ║")
print("╚══════════════════════════════════════════╝")

# Cell 6: Test Cell
This cell sends a test request to the /health endpoint to make sure everything is working before you open your mobile app.

In [ ]:
import requests

# Test the health endpoint
response = requests.get(f"{public_url}/health")
if response.status_code == 200:
    print("✅ Server is healthy!")
    print("Response:", response.json())
else:
    print("❌ Server test failed!")
    print("Status:", response.status_code)
    print("Response:", response.text)